In [ ]:
import pandas as pd
from os.path import join
import warnings, os
warnings.simplefilter("ignore")
import functions
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font

# 출력 옵션
pd.options.display.float_format = '{:,.0f}'.format

deposit_dtype = functions.deposit_dtype

In [ ]:
##################################################
# 입금구분 옵션
입금구분분석 = True

# 입력 파일
wd = r"C:\Users\DATA\Desktop"
fn_key_list = "p4_전체.xlsx"
fn_deposit = "입금조회새창.xlsx"
##################################################

# 읽기
key_list = pd.read_excel(join(wd, fn_key_list), dtype=str)
deposit = pd.read_excel(join(wd, fn_deposit), dtype=deposit_dtype)

In [ ]:
# 입금 정리
입금 = deposit[~deposit.입금구분.str.contains("부채|증명|해지|환매|매각|기타")].copy()
입금["입금월"] = 입금.입금일.str[:7]

# 원본 피벗 (계좌별 월별)
월별입금 = 입금.pivot_table(index='계좌키', columns='입금월', values='입금합계', aggfunc='sum')
월별입금.columns = [pd.to_datetime(c, format='%Y-%m').strftime('%Y년%m월') if isinstance(c, str) else c for c in 월별입금.columns]

merged = pd.merge(key_list, 월별입금, on='계좌키', how='left')
merged[월별입금.columns] = merged[월별입금.columns].fillna(0)

# 입금구분별 피벗
if 입금구분분석:
    def summary_pivot(df, label):
        p = df.pivot_table(index='매각구분', values=[c for c in df.columns if '년' in str(c)], aggfunc='sum').reset_index()
        p.insert(0, '구분', label)
        total = {'구분': label, '매각구분': '계'}
        for c in p.columns[2:]:
            total[c] = p[c].sum()
        p = pd.concat([p, pd.DataFrame([total])], ignore_index=True)
        return p

    summary_신용회복 = summary_pivot(신용회복, '신용회복') if '신용회복' in globals() else pd.DataFrame()
    summary_개인회생 = summary_pivot(개인회생, '개인회생') if '개인회생' in globals() else pd.DataFrame()
    summary_그외 = summary_pivot(그외, '그외') if '그외' in globals() else pd.DataFrame()
    summary_overall = summary_pivot(merged, '전체')
    summary_combined = pd.concat([x for x in [summary_overall, summary_신용회복, summary_개인회생, summary_그외] if not x.empty], ignore_index=True)
else:
    summary_combined = summary_pivot(merged, '전체')

In [ ]:
output_file = join(wd, os.path.splitext(fn_key_list)[0] + "_월별 입금구분별 회수내역.xlsx")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    summary_combined.to_excel(writer, sheet_name='요약', index=False)
    merged.to_excel(writer, sheet_name='전체합계', index=False)
    if 입금구분분석:
        신용회복.to_excel(writer, sheet_name='신용회복', index=False)
        개인회생.to_excel(writer, sheet_name='개인회생', index=False)
        그외.to_excel(writer, sheet_name='그외', index=False)

# 서식 처리(글꼴 10, 숫자 3자리)
wb = load_workbook(output_file)
font_size_10 = Font(size=10)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for row in ws.iter_rows():
        for cell in row:
            cell.font = font_size_10
            if isinstance(cell.value, (int, float)) and cell.value is not None:
                cell.number_format = '#,##0'
wb.save(output_file)

print(f"✓ 저장 완료: {output_file}")

✓ 저장 완료: C:\Users\DATA\Desktop\p4_전체_월별 입금구분별 회수내역.xlsx
